# NILM – Semaine 1 : prise en main des données

Objectif : charger un jeu de données de consommation (UK-DALE ou REDD), voir la courbe de la maison entière et celles de quelques appareils, et savoir quels appareils pèsent le plus.

Ce notebook n'a besoin que de `pandas`, `numpy` et `matplotlib`. Il n'utilise pas NILMTK, dont l'installation pose souvent des problèmes de versions : on s'en servira à la semaine 2 si besoin.

**Mode démo :** si `DATA_DIR` n'existe pas, le notebook génère des données **synthétiques** pour tester le code. Elles ne représentent pas une vraie maison. Remplace-les par un vrai jeu de données dès que tu l'as.

## 1. Configuration
Format attendu (celui des fichiers `.dat` de REDD et du téléchargement brut d'UK-DALE) : un dossier par maison, des fichiers `channel_N.dat` avec une ligne par mesure (`horodatage_unix puissance ...`), et un fichier `labels.dat` qui associe chaque numéro de canal à un nom (`N nom`).
Vérifie ces formats sur ton téléchargement : si les colonnes diffèrent, adapte `read_channel`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

DATA_DIR = Path("data/house_1")   # dossier d'une maison : channel_1.dat, labels.dat, ...
MAINS_NAMES = ("mains", "aggregate", "site_meter")   # noms de canaux considérés comme "maison entière"
RESAMPLE = "1min"                 # pas de temps pour l'exploration
plt.rcParams["figure.figsize"] = (11, 3.2)
plt.rcParams["axes.grid"] = True

## 2. Données de démonstration (utilisées seulement si `DATA_DIR` est absent)

In [ ]:
def make_demo(path, days=3, step_s=6, seed=0):
    """Écrit une fausse maison au même format que REDD / UK-DALE (données SYNTHÉTIQUES)."""
    rng = np.random.default_rng(seed)
    path.mkdir(parents=True, exist_ok=True)
    t = np.arange(0, days * 86400, step_s)
    hour = (t / 3600) % 24
    base = 60 + rng.normal(0, 3, t.size)                                   # veille permanente
    fridge = np.where(((t // 900) % 2 == 0), 90.0, 0.0)                    # cycles de ~15 min
    heater = np.where(((hour > 2) & (hour < 4.5)) | ((hour > 12) & (hour < 13)), 2100.0, 0.0)   # chauffe-eau
    washer = np.zeros_like(t, dtype=float)
    for d in range(days):
        s = d * 86400 + 19.5 * 3600
        washer[(t >= s) & (t < s + 3600)] = 500 + 1800 * ((t[(t >= s) & (t < s + 3600)] - s) < 1200)
    ts0 = 1_700_000_000
    chans = {"mains": base + fridge + heater + washer, "fridge": fridge + 5, "water_heater": heater, "washing_machine": washer}
    labels = []
    for i, (name, p) in enumerate(chans.items(), start=1):
        pd.DataFrame({"t": ts0 + t, "w": np.maximum(p, 0).round(1)}).to_csv(path / f"channel_{i}.dat", sep=" ", header=False, index=False)
        labels.append(f"{i} {name}")
    (path / "labels.dat").write_text("\n".join(labels) + "\n")

if not DATA_DIR.exists():
    print("DATA_DIR introuvable : génération de données de DÉMO synthétiques.")
    DATA_DIR = Path("demo_house")
    make_demo(DATA_DIR)
print("Dossier utilisé :", DATA_DIR.resolve())

## 3. Chargement

In [ ]:
def read_labels(folder):
    labels = {}
    for line in (folder / "labels.dat").read_text().splitlines():
        parts = line.split(maxsplit=1)
        if len(parts) == 2:
            labels[int(parts[0])] = parts[1].strip()
    return labels

def read_channel(folder, ch):
    """Lit channel_N.dat : colonne 0 = horodatage unix, colonne 1 = puissance (W)."""
    df = pd.read_csv(folder / f"channel_{ch}.dat", sep=r"\s+", header=None, usecols=[0, 1], names=["t", "w"])
    df.index = pd.to_datetime(df["t"], unit="s")
    return df["w"].astype(float)

labels = read_labels(DATA_DIR)
print(labels)
series = {}
for ch, name in labels.items():
    try:
        series[name] = read_channel(DATA_DIR, ch).resample(RESAMPLE).mean()
    except FileNotFoundError:
        print("canal manquant :", ch, name)
data = pd.DataFrame(series)
print(data.shape, "|", data.index.min(), "->", data.index.max())
data.head()

## 4. La maison entière : la courbe qu'on va devoir « démonter »

In [ ]:
mains_cols = [c for c in data.columns if c.lower() in MAINS_NAMES or c.lower().startswith("mains")]
print("Canaux maison entière :", mains_cols)
mains = data[mains_cols].sum(axis=1) if len(mains_cols) > 1 else data[mains_cols[0]]   # REDD : 2 phases à additionner
ax = mains.plot(title="Puissance de la maison entière (W)", color="#0E7C86")
ax.set_ylabel("W"); plt.show()

## 5. Un jour, maison entière et appareils

In [ ]:
appliances = [c for c in data.columns if c not in mains_cols]
day = data.index.min().normalize() + pd.Timedelta(days=1)       # un jour complet (le 2e)
d = data.loc[day: day + pd.Timedelta(days=1)]
fig, axes = plt.subplots(1 + min(3, len(appliances)), 1, figsize=(11, 8), sharex=True)
axes[0].plot(d.index, d[mains_cols].sum(axis=1), color="#0E7C86"); axes[0].set_title("Maison entière (W)")
top = data[appliances].sum().sort_values(ascending=False).index[:3]   # les 3 plus gros en énergie
for ax, name in zip(axes[1:], top):
    ax.plot(d.index, d[name], color="#E8A000"); ax.set_title(name + " (W)")
plt.tight_layout(); plt.show()

## 6. Bilan d'énergie : quels appareils compter ?
Pour la flexibilité, ce qui compte, ce sont les gros consommateurs qu'on peut décaler. Ce tableau donne la part de chaque appareil dans la consommation.

In [ ]:
hours = pd.Timedelta(RESAMPLE).total_seconds() / 3600
energy_kwh = (data.fillna(0) * hours).sum() / 1000
summary = pd.DataFrame({"kWh": energy_kwh})
summary["part de la maison (%)"] = 100 * summary["kWh"] / energy_kwh[mains_cols].sum()
summary["puissance max (W)"] = data.max()
summary["% du temps allumé (>10 W)"] = 100 * (data > 10).mean()
summary.sort_values("kWh", ascending=False).round(1)

## 7. Événements : les marches de la courbe
Le NILM repère les sauts de puissance quand un appareil s'allume ou s'éteint. Voici les plus gros sauts de la maison entière.

In [ ]:
step = mains.diff()
events = step[step.abs() > 300].dropna()          # seuil en W : à ajuster
print(len(events), "événements > 300 W")
events.sort_values(key=lambda s: -s.abs()).head(10).round(0)

In [ ]:
fig, ax = plt.subplots()
ax.hist(events.values, bins=40, color="#0E7C86")
ax.set_title("Taille des sauts de puissance (W) : chaque pic est une famille d'appareils")
plt.show()

## 8. À noter pour la suite
- [ ] Quels appareils sont présents dans ton jeu de données, et en combien de maisons ?
- [ ] Y a-t-il un chauffe-eau ? Sinon, le rôle « gros consommateur décalable » sera tenu par un autre appareil (lave-linge, four...).
- [ ] Quelle fréquence d'échantillonnage pour la maison entière et pour les appareils ? C'est elle qui limite ce que le modèle peut voir.
- [ ] Périodes avec des trous dans les données : à exclure de l'entraînement.